In [320]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [321]:
data = pd.read_csv('processed_data_tennis_scaled_v2.csv')
data

,has_secondary_sport_education,has_higher_sport_education,sport_master,sport_master_kandidate,is_champion,work_experience,master_id,rang_id,docs_verified,rating,...,photos_count,is_male,has_price,years_on_profi_ru,price_in_thousands,cur_score_percents,work_experience_sq,reputation_density,reputation_volume_interaction,champion_experience_inter
0,1,0,0,0,0,4.0,https://profi.ru/profile/AbelkhairovaSV/,86.0,1,5.0,...,10,0,1,2.416667,3.2,0.504935,16.0,1.463415,8.958797,0.000000
1,1,0,0,0,0,19.0,https://profi.ru/profile/AbramovMB/,111.0,0,0.0,...,0,1,0,0.166667,NaN,0.504337,361.0,0.000000,0.000000,0.000000
2,0,1,1,0,1,8.0,https://profi.ru/profile/AbyasovaAN/,100.0,0,0.0,...,0,0,0,2.583333,NaN,0.503504,64.0,0.000000,0.000000,2.583333
3,1,0,0,0,0,2.0,https://profi.ru/profile/AfanasyevBA4/,144.0,1,0.0,...,8,1,1,1.250000,2.0,0.503303,4.0,0.000000,0.000000,0.000000
4,1,0,1,0,0,3.0,https://profi.ru/profile/AgureyevaKM2/,75.0,1,5.0,...,0,0,1,0.666667,3.0,0.505121,9.0,2.400000,8.047190,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407,1,0,0,1,0,5.0,https://profi.ru/profile/ZhirkovaVA10/,136.0,0,0.0,...,2,0,1,0.166667,3.0,0.500943,25.0,0.000000,0.000000,0.000000
408,1,0,0,0,0,13.0,https://profi.ru/profile/ZhuchkovaVS3/,136.0,0,0.0,...,0,0,0,0.750000,NaN,0.500192,169.0,0.000000,0.000000,0.000000
409,1,0,1,0,1,9.0,https://profi.ru/profile/ZhukovMA78/,103.0,1,0.0,...,3,1,0,0.083333,NaN,0.504611,81.0,0.000000,0.000000,0.083333
410,0,1,1,0,1,4.0,https://profi.ru/profile/ZykovaDS2/,84.0,0,5.0,...,2,0,1,2.750000,5.0,0.504794,16.0,0.266667,3.465736,2.750000


In [322]:
## делаю безлайн только на данных с ценой
data_priced = data[data['has_price'] == 1].copy()

In [323]:
y = data_priced['price_in_thousands']
X = sm.add_constant(data_priced[['has_secondary_sport_education', 'has_higher_sport_education',
       'sport_master', 'sport_master_kandidate', 'is_champion',
       'work_experience', 'rang_id', 'docs_verified', 'rating',
       'reviews_count', 'is_recomended', 'photos_count', 'is_male', 'years_on_profi_ru', 'cur_score_percents', 'work_experience_sq', 'reputation_density',
       'reputation_volume_interaction', 'champion_experience_inter']])

In [324]:
# Расчет VIF для изначальных данных
vif_data = pd.DataFrame()
vif_data["Переменная"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

                       Переменная         VIF
0                           const  284.383586
1   has_secondary_sport_education    8.707307
2      has_higher_sport_education    9.082903
3                    sport_master    1.858797
4          sport_master_kandidate    1.238310
5                     is_champion    3.168067
6                 work_experience    9.997958
7                         rang_id    1.740212
8                   docs_verified    1.264301
9                          rating    3.777690
10                  reviews_count    5.448035
11                  is_recomended    1.498524
12                   photos_count    1.152380
13                        is_male    1.300049
14              years_on_profi_ru    2.843480
15             cur_score_percents    1.086843
16             work_experience_sq    9.116383
17             reputation_density    5.422149
18  reputation_volume_interaction   11.181601
19      champion_experience_inter    3.238861


### Логика такая - для `reputation_volume_interaction`  VIF слишком большой, его удалим


### Видим, что почти у всех есть какое-то образование, признаки сильно коррелируют, так что удалим  признак `has_higher_sport_education`

In [325]:
(X['has_secondary_sport_education'] + X['has_higher_sport_education']).mean()

np.float64(0.9655172413793104)

### `rang_id`, `cur_score_percents` уберем, т.к признаки не подходят по смыслу(экономическая интуиция)

In [326]:
data_priced.drop(['has_secondary_sport_education', 'reputation_volume_interaction', 'rang_id', 'cur_score_percents'], axis=1, inplace=True)

In [ ]:
model = sm.OLS(y, X).fit()

beta_exp = model.params['work_experience']
beta_exp2 = model.params['work_experience_sq']

peak = -beta_exp / (2 * beta_exp2)
## центрирование данных
data_priced['exp_centered'] = data_priced['work_experience'] - peak
data_priced['exp_centered2'] = data_priced['exp_centered'] ** 2

y = data_priced['price_in_thousands']
X = sm.add_constant(data_priced[['has_higher_sport_education',
       'sport_master', 'sport_master_kandidate', 'is_champion',
       'exp_centered', 'docs_verified', 'rating',
       'reviews_count', 'is_recomended', 'photos_count', 'is_male', 'years_on_profi_ru', 'exp_centered2', 'reputation_density',
       'champion_experience_inter']])

In [328]:
# Расчет VIF для центрированных данных
vif_data = pd.DataFrame()
vif_data["Переменная"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

                    Переменная        VIF
0                        const  12.337090
1   has_higher_sport_education   1.214726
2                 sport_master   1.754891
3       sport_master_kandidate   1.201156
4                  is_champion   3.128500
5                 exp_centered   1.524714
6                docs_verified   1.211409
7                       rating   1.549468
8                reviews_count   4.507510
9                is_recomended   1.305600
10                photos_count   1.124300
11                     is_male   1.082138
12           years_on_profi_ru   2.620070
13               exp_centered2   1.331674
14          reputation_density   4.340339
15   champion_experience_inter   3.184013


###  Вывод: все VIF меньше 5, с мультиколлинеарность не сильная

## Уберем самые незначимые фичи по p-value, с учетом того, что после удаления фичи AIC и BIC должны вырасти

In [329]:
## запишем колонки, которые будем дропать
cols_to_drop = []

In [330]:
base_model_res = sm.OLS(y, X).fit()
worst_p = base_model_res.pvalues.drop('const').max()
worst_var = base_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии базовой модели
print(f"Текущий AIC: {base_model_res.aic:.2f} | BIC: {base_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1058.71 | BIC: 1118.95
Худшая переменная: 'rating' (p-value = 0.8861)
------------------------------


In [331]:
base_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.094
Method:                 Least Squares   F-statistic:                     3.190
Date:                Thu, 07 May 2026   Prob (F-statistic):           6.72e-05
Time:                        18:25:06   Log-Likelihood:                -513.35
No. Observations:                 319   AIC:                             1059.
Df Residuals:                     303   BIC:                             1119.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7905      0.244     11.432      0.000       2.310       3.271
has_higher_sport_education    -0.1845      0.154     -1.201      0.231      -0.487       0.118
sport_master                   0.6317      0.185      3.410      0.001       0.267       0.996
sport_master_kandidate         0.3047      0.269      1.132      0.258      -0.225       0.834
is_champion                    0.4312      0.270      1.599      0.111      -0.099       0.962
exp_centered                  -0.0021      0.011     -0.200      0.841      -0.023       0.019
docs_verified                 -0.1300      0.166     -0.784      0.434      -0.456       0.196
rating                        -0.0055      0.038     -0.143      0.886      -0.081       0.070
reviews_count                 -0.0026      0.007     -0.363      0.717      -0.017       0.012
is_recomended                  0.1390      0.251      0.554      0.580      -0.355       0.633
photos_count                   0.0372      0.011      3.250      0.001       0.015       0.060
is_male                       -0.1466      0.146     -1.005      0.316      -0.434       0.141
years_on_profi_ru              0.0273      0.028      0.974      0.331      -0.028       0.083
exp_centered2                 -0.0012      0.001     -1.519      0.130      -0.003       0.000
reputation_density             0.0180      0.045      0.403      0.688      -0.070       0.106
champion_experience_inter     -0.0418      0.038     -1.087      0.278      -0.118       0.034
==============================================================================
Omnibus:                      150.483   Durbin-Watson:                   2.020
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              946.486
Skew:                           1.861   Prob(JB):                    2.97e-206
Kurtosis:                      10.573   Cond. No.                         662.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Уберем переменную `rating`

In [332]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1056.73 | BIC: 1113.21
Худшая переменная: 'exp_centered' (p-value = 0.8512)
------------------------------


In [333]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.097
Method:                 Least Squares   F-statistic:                     3.427
Date:                Thu, 07 May 2026   Prob (F-statistic):           3.50e-05
Time:                        18:25:06   Log-Likelihood:                -513.37
No. Observations:                 319   AIC:                             1057.
Df Residuals:                     304   BIC:                             1113.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7866      0.242     11.506      0.000       2.310       3.263
has_higher_sport_education    -0.1860      0.153     -1.216      0.225      -0.487       0.115
sport_master                   0.6285      0.184      3.423      0.001       0.267       0.990
sport_master_kandidate         0.3036      0.269      1.130      0.259      -0.225       0.832
is_champion                    0.4320      0.269      1.605      0.109      -0.098       0.962
exp_centered                  -0.0020      0.011     -0.188      0.851      -0.023       0.019
docs_verified                 -0.1348      0.162     -0.831      0.407      -0.454       0.184
reviews_count                 -0.0025      0.007     -0.346      0.730      -0.016       0.012
is_recomended                  0.1376      0.250      0.550      0.583      -0.355       0.630
photos_count                   0.0372      0.011      3.254      0.001       0.015       0.060
is_male                       -0.1456      0.145     -1.000      0.318      -0.432       0.141
years_on_profi_ru              0.0261      0.027      0.978      0.329      -0.026       0.079
exp_centered2                 -0.0012      0.001     -1.538      0.125      -0.003       0.000
reputation_density             0.0159      0.042      0.377      0.706      -0.067       0.099
champion_experience_inter     -0.0419      0.038     -1.091      0.276      -0.118       0.034
==============================================================================
Omnibus:                      150.598   Durbin-Watson:                   2.021
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              949.152
Skew:                           1.862   Prob(JB):                    7.84e-207
Kurtosis:                      10.586   Cond. No.                         662.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `rating`, *AIC* и *BIC* упали

### Уберем переменную `exp_centered`

In [334]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1054.77 | BIC: 1107.48
Худшая переменная: 'reviews_count' (p-value = 0.7068)
------------------------------


In [335]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.099
Method:                 Least Squares   F-statistic:                     3.700
Date:                Thu, 07 May 2026   Prob (F-statistic):           1.76e-05
Time:                        18:25:06   Log-Likelihood:                -513.38
No. Observations:                 319   AIC:                             1055.
Df Residuals:                     305   BIC:                             1107.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8023      0.227     12.352      0.000       2.356       3.249
has_higher_sport_education    -0.1922      0.149     -1.288      0.199      -0.486       0.101
sport_master                   0.6253      0.183      3.425      0.001       0.266       0.985
sport_master_kandidate         0.3050      0.268      1.138      0.256      -0.223       0.833
is_champion                    0.4349      0.268      1.621      0.106      -0.093       0.963
docs_verified                 -0.1342      0.162     -0.829      0.408      -0.453       0.184
reviews_count                 -0.0027      0.007     -0.376      0.707      -0.017       0.011
is_recomended                  0.1401      0.250      0.561      0.575      -0.351       0.631
photos_count                   0.0373      0.011      3.273      0.001       0.015       0.060
is_male                       -0.1494      0.144     -1.038      0.300      -0.432       0.134
years_on_profi_ru              0.0254      0.026      0.962      0.337      -0.027       0.077
exp_centered2                 -0.0012      0.001     -1.566      0.118      -0.003       0.000
reputation_density             0.0169      0.042      0.404      0.686      -0.065       0.099
champion_experience_inter     -0.0420      0.038     -1.095      0.274      -0.118       0.033
==============================================================================
Omnibus:                      149.690   Durbin-Watson:                   2.022
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              932.496
Skew:                           1.853   Prob(JB):                    3.24e-203
Kurtosis:                      10.512   Cond. No.                         656.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `exp_centered`, *AIC* и *BIC* упали

### Уберем переменную `reviews_count`

In [336]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1052.92 | BIC: 1101.86
Худшая переменная: 'reputation_density' (p-value = 0.8671)
------------------------------


In [337]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.102
Method:                 Least Squares   F-statistic:                     4.007
Date:                Thu, 07 May 2026   Prob (F-statistic):           8.86e-06
Time:                        18:25:06   Log-Likelihood:                -513.46
No. Observations:                 319   AIC:                             1053.
Df Residuals:                     306   BIC:                             1102.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8154      0.224     12.574      0.000       2.375       3.256
has_higher_sport_education    -0.1901      0.149     -1.277      0.203      -0.483       0.103
sport_master                   0.6261      0.182      3.435      0.001       0.267       0.985
sport_master_kandidate         0.3054      0.268      1.141      0.255      -0.221       0.832
is_champion                    0.4377      0.268      1.635      0.103      -0.089       0.965
docs_verified                 -0.1314      0.161     -0.813      0.417      -0.449       0.186
is_recomended                  0.1183      0.242      0.488      0.626      -0.359       0.595
photos_count                   0.0372      0.011      3.271      0.001       0.015       0.060
is_male                       -0.1486      0.144     -1.034      0.302      -0.431       0.134
years_on_profi_ru              0.0217      0.024      0.887      0.376      -0.026       0.070
exp_centered2                 -0.0012      0.001     -1.584      0.114      -0.003       0.000
reputation_density             0.0040      0.024      0.168      0.867      -0.043       0.051
champion_experience_inter     -0.0428      0.038     -1.118      0.265      -0.118       0.033
==============================================================================
Omnibus:                      148.973   Durbin-Watson:                   2.020
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              926.748
Skew:                           1.842   Prob(JB):                    5.75e-202
Kurtosis:                      10.493   Cond. No.                         654.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `reviews_count`, *AIC* и *BIC* упали

### Уберем переменную `reputation_density`

In [338]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1050.95 | BIC: 1096.13
Худшая переменная: 'is_recomended' (p-value = 0.5695)
------------------------------


In [339]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.136
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     4.383
Date:                Thu, 07 May 2026   Prob (F-statistic):           4.08e-06
Time:                        18:25:06   Log-Likelihood:                -513.47
No. Observations:                 319   AIC:                             1051.
Df Residuals:                     307   BIC:                             1096.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8168      0.223     12.610      0.000       2.377       3.256
has_higher_sport_education    -0.1907      0.149     -1.283      0.200      -0.483       0.102
sport_master                   0.6264      0.182      3.442      0.001       0.268       0.984
sport_master_kandidate         0.3100      0.266      1.166      0.245      -0.213       0.833
is_champion                    0.4361      0.267      1.632      0.104      -0.090       0.962
docs_verified                 -0.1271      0.159     -0.798      0.425      -0.440       0.186
is_recomended                  0.1309      0.230      0.569      0.569      -0.322       0.583
photos_count                   0.0377      0.011      3.425      0.001       0.016       0.059
is_male                       -0.1461      0.143     -1.024      0.307      -0.427       0.135
years_on_profi_ru              0.0213      0.024      0.877      0.381      -0.027       0.069
exp_centered2                 -0.0012      0.001     -1.583      0.115      -0.003       0.000
champion_experience_inter     -0.0429      0.038     -1.122      0.263      -0.118       0.032
==============================================================================
Omnibus:                      148.519   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              918.296
Skew:                           1.838   Prob(JB):                    3.93e-200
Kurtosis:                      10.455   Cond. No.                         650.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `reputation_density`, *AIC* и *BIC* упали

### Уберем переменную `is_recomended`

In [340]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1049.28 | BIC: 1090.70
Худшая переменная: 'docs_verified' (p-value = 0.4606)
------------------------------


In [341]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.107
Method:                 Least Squares   F-statistic:                     4.799
Date:                Thu, 07 May 2026   Prob (F-statistic):           2.02e-06
Time:                        18:25:06   Log-Likelihood:                -513.64
No. Observations:                 319   AIC:                             1049.
Df Residuals:                     308   BIC:                             1091.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8131      0.223     12.612      0.000       2.374       3.252
has_higher_sport_education    -0.1915      0.148     -1.290      0.198      -0.484       0.101
sport_master                   0.6279      0.182      3.454      0.001       0.270       0.986
sport_master_kandidate         0.2978      0.265      1.125      0.262      -0.223       0.819
is_champion                    0.4377      0.267      1.640      0.102      -0.087       0.963
docs_verified                 -0.1167      0.158     -0.739      0.461      -0.428       0.194
photos_count                   0.0381      0.011      3.473      0.001       0.017       0.060
is_male                       -0.1450      0.142     -1.018      0.310      -0.425       0.135
years_on_profi_ru              0.0235      0.024      0.979      0.328      -0.024       0.071
exp_centered2                 -0.0012      0.001     -1.602      0.110      -0.003       0.000
champion_experience_inter     -0.0429      0.038     -1.125      0.261      -0.118       0.032
==============================================================================
Omnibus:                      147.330   Durbin-Watson:                   2.016
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              896.186
Skew:                           1.826   Prob(JB):                    2.49e-195
Kurtosis:                      10.354   Cond. No.                         650.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `is_recomended`, *AIC* и *BIC* упали

### Уберем переменную `docs_verified`

In [342]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1047.85 | BIC: 1085.50
Худшая переменная: 'years_on_profi_ru' (p-value = 0.3837)
------------------------------


In [343]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.108
Method:                 Least Squares   F-statistic:                     5.280
Date:                Thu, 07 May 2026   Prob (F-statistic):           1.05e-06
Time:                        18:25:06   Log-Likelihood:                -513.92
No. Observations:                 319   AIC:                             1048.
Df Residuals:                     309   BIC:                             1085.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7480      0.205     13.420      0.000       2.345       3.151
has_higher_sport_education    -0.1910      0.148     -1.288      0.199      -0.483       0.101
sport_master                   0.6332      0.181      3.489      0.001       0.276       0.990
sport_master_kandidate         0.3186      0.263      1.211      0.227      -0.199       0.836
is_champion                    0.4300      0.266      1.614      0.108      -0.094       0.954
photos_count                   0.0368      0.011      3.400      0.001       0.015       0.058
is_male                       -0.1476      0.142     -1.037      0.300      -0.428       0.132
years_on_profi_ru              0.0206      0.024      0.872      0.384      -0.026       0.067
exp_centered2                 -0.0012      0.001     -1.564      0.119      -0.003       0.000
champion_experience_inter     -0.0430      0.038     -1.128      0.260      -0.118       0.032
==============================================================================
Omnibus:                      146.189   Durbin-Watson:                   2.028
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              867.609
Skew:                           1.819   Prob(JB):                    3.99e-189
Kurtosis:                      10.214   Cond. No.                         638.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `docs_verified`, *AIC* и *BIC* упали

### Уберем переменную `years_on_profi_ru`

In [344]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1046.63 | BIC: 1080.52
Худшая переменная: 'champion_experience_inter' (p-value = 0.4347)
------------------------------


In [345]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.131
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     5.849
Date:                Thu, 07 May 2026   Prob (F-statistic):           5.70e-07
Time:                        18:25:06   Log-Likelihood:                -514.32
No. Observations:                 319   AIC:                             1047.
Df Residuals:                     310   BIC:                             1081.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8219      0.186     15.142      0.000       2.455       3.189
has_higher_sport_education    -0.1587      0.144     -1.105      0.270      -0.441       0.124
sport_master                   0.6706      0.176      3.804      0.000       0.324       1.017
sport_master_kandidate         0.3313      0.263      1.262      0.208      -0.185       0.848
is_champion                    0.3256      0.238      1.368      0.172      -0.143       0.794
photos_count                   0.0370      0.011      3.425      0.001       0.016       0.058
is_male                       -0.1317      0.141     -0.933      0.351      -0.409       0.146
exp_centered2                 -0.0013      0.001     -1.886      0.060      -0.003    5.82e-05
champion_experience_inter     -0.0252      0.032     -0.782      0.435      -0.088       0.038
==============================================================================
Omnibus:                      144.791   Durbin-Watson:                   2.027
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              838.098
Skew:                           1.808   Prob(JB):                    1.02e-182
Kurtosis:                      10.069   Cond. No.                         625.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `years_on_profi_ru`, *AIC* и *BIC* упали

### Уберем переменную `champion_experience_inter`

In [346]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 1045.26 | BIC: 1075.38
Худшая переменная: 'is_male' (p-value = 0.3078)
------------------------------


In [347]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.129
Model:                            OLS   Adj. R-squared:                  0.110
Method:                 Least Squares   F-statistic:                     6.605
Date:                Thu, 07 May 2026   Prob (F-statistic):           2.73e-07
Time:                        18:25:06   Log-Likelihood:                -514.63
No. Observations:                 319   AIC:                             1045.
Df Residuals:                     311   BIC:                             1075.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8210      0.186     15.147      0.000       2.455       3.188
has_higher_sport_education    -0.1581      0.143     -1.102      0.271      -0.440       0.124
sport_master                   0.6674      0.176      3.789      0.000       0.321       1.014
sport_master_kandidate         0.3393      0.262      1.294      0.197      -0.177       0.855
is_champion                    0.1994      0.175      1.140      0.255      -0.145       0.544
photos_count                   0.0361      0.011      3.359      0.001       0.015       0.057
is_male                       -0.1433      0.140     -1.021      0.308      -0.419       0.133
exp_centered2                 -0.0012      0.001     -1.762      0.079      -0.003       0.000
==============================================================================
Omnibus:                      144.871   Durbin-Watson:                   2.020
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              842.122
Skew:                           1.808   Prob(JB):                    1.37e-183
Kurtosis:                      10.091   Cond. No.                         619.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `champion_experience_inter`, *AIC* и *BIC* упали

## пока стоп, остановимся с удалением фичей, сделаем тесты на спецификацию и еще раз сделаем центривание `work_experience`

In [ ]:
data_left = X.drop(cols_to_drop, axis=1)

In [350]:
rest_features = data_left.columns
rest_features

Index(['const', 'has_higher_sport_education', 'sport_master',
       'sport_master_kandidate', 'is_champion', 'photos_count', 'is_male',
       'exp_centered2'],
      dtype='object')